# 03 · Experiment tracking & DOE

Runs the Stage 2 forecasters through the Stage 3 harness: the **DOE** matrix expands the config
into a grid of variants, and `run_experiment` drives each (variant × enabled model), logging
params, MAE/RMSE vs the held-out **TEST** actuals, and prediction/config artifacts to **Vertex AI
Experiments** + GCS.

- This is the **first live execution** of the forecasters (TimesFM downloads the 200M checkpoint;
  BQML submits real BigQuery jobs). Needs ADC auth + network.
- BQML reads the `train`/`infer` tables, built once below from the prepped table.
- Artifacts land under `gs://<bucket>/experiments/<experiment_name>/<run>/`.
- Run with the **geapTimes (uv 3.11)** kernel.

### Colab (optional)

To run in Colab instead of the local `uv` kernel, uncomment the cell below to install the
package and authenticate. Keep it commented for the local flow.

In [ ]:
# !pip install -q -e ..
# from google.colab import auth; auth.authenticate_user()

In [ ]:
from geaptimes.naming import table_names
from geaptimes.schemas import ExperimentConfig

cfg = ExperimentConfig.from_yaml("../config/base_config.yaml")
enabled = [(m.name, m.params.type) for m in cfg.models if m.enabled]
print("enabled models:", enabled)
print("DOE axes:", cfg.doe.axes)
print("experiment:", cfg.execution.experiment_name, "| horizon:", cfg.forecast.horizon)

### Build the BQML input tables (one-time)

`train` = rows where `splits != 'TEST'`; `infer` = the held-out TEST window with the target nulled
(real known-future regressors). TimesFM needs neither — it reads the prepped table directly.

In [ ]:
from google.cloud import bigquery

from geaptimes.data.queries import build_infer_query, build_train_query

client = bigquery.Client(project=cfg.project.id)
names = table_names(cfg)
ds = f"{cfg.project.id}.{cfg.project.bq_dataset}"
prepped_id = f"{ds}.{names['prepped']}"

for label, sql in [
    ("train", build_train_query(cfg.data, prepped_id, f"{ds}.{names['train']}")),
    ("infer", build_infer_query(cfg.data, prepped_id, f"{ds}.{names['infer']}")),
]:
    client.query(sql).result()
    print("built", label, "->", names[label])

### Run the experiment

One tracked `ExperimentRun` per (DOE variant × enabled model). Each `RunRecord` carries the run
name, metrics, and the GCS artifact URI.

In [ ]:
import pandas as pd

from geaptimes.experiment.runner import run_experiment

records = run_experiment(cfg)
summary = pd.DataFrame(
    [
        {
            "model": r.model,
            "run_name": r.run_name,
            "mae": r.metrics["mae"],
            "rmse": r.metrics["rmse"],
            "n_points": r.metrics["n_points"],
            "artifact_uri": r.artifact_uri,
        }
        for r in records
    ]
)
summary

### Plot forecast vs TEST actuals

For one enabled model and one station, with the standardized `q10–q90` band. Re-runs `predict()`
for the chosen model to recover the prediction frame (the run loop logs it to GCS rather than
returning it).

In [ ]:
import matplotlib.pyplot as plt

from geaptimes.models.factory import ForecastFactory

model_cfg = next(m for m in cfg.models if m.enabled)
forecaster = ForecastFactory.create(model_cfg, cfg)
forecaster.fit()
pred_all = forecaster.predict().predictions

data = cfg.data
sql = (
    f"SELECT {data.series_column} AS series, {data.time_column} AS date, "
    f"{data.target_column} AS actual FROM `{prepped_id}` WHERE splits = 'TEST'"
)
actuals = client.query(sql).to_dataframe()

station = pred_all["series"].iloc[0]
pred = pred_all[pred_all["series"] == station].sort_values("date")
act = actuals[actuals["series"] == station].sort_values("date")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(act["date"], act["actual"], "o-", label="actual (TEST)", color="black")
ax.plot(pred["date"], pred["forecast"], "s--", label="forecast", color="tab:blue")
ax.fill_between(pred["date"], pred["q10"], pred["q90"], alpha=0.2, color="tab:blue", label="q10-q90")
ax.set_title(f"{model_cfg.name} — {station}")
ax.set_xlabel("date")
ax.set_ylabel(cfg.data.target_column)
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()